# 123 — Handoffs y transferencia de contexto

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** Un payload válido (lo importante es la estructura y la separación
hechos/hipótesis):

```json
{
  "from_agent": "data-analysis", "to_agent": "compliance",
  "reason": "datos personales detectados: fuera de mi ámbito",
  "goal": "decidir base legal y anonimización antes de continuar el análisis",
  "state": {
    "verified_facts": ["columna 'email' con 12k valores", "no hay consentimiento registrado en el esquema"],
    "work_done": ["perfilado de columnas", "análisis detenido en el paso 2/5"],
    "work_remaining": ["decidir anonimización", "reanudar análisis"],
    "constraints": ["no mover datos fuera de la región", "SLA 24h"]
  }
}
```

**Ejercicio 2.** Ratio = 260/4200 ≈ **0.062** (≈16× de compresión). Ahorro por salto =
(4200 − 260) × 3e-6 ≈ 0.0118 USD; en 5 saltos ≈ **0.059 USD** por tarea. Parece poco,
pero escala linealmente con el volumen y, sobre todo, el coste real evitado es
cualitativo: 5 × 4 200 = 21 000 tokens de historial repetido sepultarían lo accionable.

**Ejercicio 3.** Basta un `set` de pares vistos; el primer par que reaparezca es la
señal de ciclo — aquí `("infra","security")` en el cuarto handoff.

**Ejercicio 4.** El dueño es el **supervisor**: formula la tarea, los workers devuelven
`{agent, score, finding}` y el control *vuelve* — es delegación tipo subagente, no
handoff. Evidencia: `result.supervisor.decision` existe (el supervisor cierra la
tarea) y ningún worker referencia a otro ni a un traspaso; `evidence` describe
consolidación, no transferencia de ownership.


In [ ]:
result = run_lab("multiagent", seed=123)
assert result["kind"] == "multiagent"
assert result["evidence"]
show(result)


In [ ]:
# Ejercicio 2
ratio = 260 / 4200
ahorro_usd = 5 * (4200 - 260) * 3e-6
print(f"ratio = {ratio:.3f} (≈{4200/260:.0f}x de compresión), ahorro ≈ {ahorro_usd:.3f} USD")

# Ejercicio 3
def primer_par_repetido(handoffs):
    vistos = set()
    for par in handoffs:
        if par in vistos:
            return par
        vistos.add(par)
    return None

cadena = [("triage", "infra"), ("infra", "security"),
          ("security", "infra"), ("infra", "security")]
print("ciclo detectado en:", primer_par_repetido(cadena))

# Ejercicio 4
result = run_lab("multiagent", seed=123)
assert result["kind"] == "multiagent"
assert "decision" in result["result"]["supervisor"]  # el supervisor cierra: delegación, no handoff
print("dueño: supervisor →", result["result"]["supervisor"])


## Reflexión

1. En el payload de ejemplo, ¿qué pasaría si `verified_facts` y las hipótesis viajaran mezcladas en un solo campo `notes`? Describe un fallo concreto que B podría cometer.
2. El laboratorio consolida workers que nunca se traspasan la tarea. ¿En qué punto del flujo supervisor→workers tendría sentido un handoff worker→worker y qué campos del payload serían críticos?
3. ¿Qué métrica registrarías para detectar que los handoffs de tu sistema están destruyendo contexto (el clásico "teléfono roto")?
